# ECON 524 — Big Data Econometrics
## Computational Assignment 3: Causal Inference with High-Dimensional Covariates (LATE and Difference-in-Differences)

**Emory University · Fall 2026**  
Instructor: Zheng Fang · Lab instructor: Joel Reyes Mora · Grader: Marcelo Ortiz-Villavicencio

**Time allowed:** two weeks from release. **Based on:** Lecture 3 (Identification of Causal Effects), §2 (Instrumental Variables and LATE, slides 13–28), §3 (Difference-in-Differences, slides 30–42), and slide 43 (*Causal Inference and Machine Learning*). The questions use the learners from Lectures 1–2 (LASSO, $\ell_1$-penalized logistic regression, regression trees, random forests) as estimators of the **nuisance functions** that the identification results depend on.

## Instructions

1. **Submit this notebook** with every cell executed from top to bottom (*Kernel → Restart & Run All*) and all output visible.
2. **Written work goes in Markdown cells** (LaTeX is fine). Derivations marked *by hand* must be typed in Markdown; a photo of handwritten work is acceptable if embedded in the notebook.
3. **Every figure or table must be followed by an interpretation** that answers the question asked, not a description of the plot. Interpretation carries roughly half the credit of each part.
4. **Reproducibility.** Fix and report random seeds. Everything runs on a laptop CPU. The random-forest Monte Carlo in Q2(b) is the most expensive step; budget about 10–20 minutes for it and use all your cores.
5. **Allowed software.** NumPy, pandas, SciPy, statsmodels, matplotlib, and scikit-learn. **Implement every IV, DiD and event-study estimator yourself** (two-stage least squares, Wald, plug-in, inverse-probability-weighted, TWFE, group-time ATT, event-study aggregation, bootstrap). You may use a library only to fit the *learners* (LASSO, logistic regression, trees, forests) and, if you wish, to check your own 2SLS or TWFE output.
6. **AI use.** Follow the course README's responsible-use framework and include the required acknowledgment statement at the end of the notebook.

**Ground rules.**
- All data are simulated from the three designs below, so you can compute every causal parameter **exactly** from the potential outcomes, which are never observed in practice. Use potential outcomes, compliance types and the true nuisance functions **only** where a question says *truth* or *oracle*. Every estimator must use observed data only: $(Z, W, D, Y)$ in Q1–Q2, $(Z, D, Y_{-1}, Y_0, Y_1)$ in Q3, and $(Z, G, Y_1,\dots,Y_8)$ in Q4.
- "Population" quantities are approximated with a very large simulated sample (sizes are given in each question; process it in chunks if memory is tight).
- Throughout, standardize covariates before fitting a LASSO or an $\ell_1$-penalized logistic regression. Random forests do not need standardization.
- **Notation.** Covariates are indexed from 1: $z_1$ is the first column of $Z$.

## Simulation designs

All three designs draw covariates as $Z \sim N(0,\Sigma)$ with Toeplitz correlation $\Sigma_{jk}=0.5^{|j-k|}$, so neighbouring covariates are correlated and distant ones nearly independent. $\Lambda(u)=1/(1+e^{-u})$ is the logistic CDF.

### DGP-A — A job-training lottery with priority odds (Q1–Q2)

A program has more applicants than seats and allocates **offers** $W$ by lottery. As in many real lotteries (e.g., the Oregon Health Insurance Experiment, where household size changed the odds of selection; Finkelstein et al., 2012), the odds depend on the applicant file, so $W$ is random **only conditional on $Z$**. Earnings are in \$1,000s.

| Component | Specification |
|---|---|
| Applicant file | $p=100$ covariates. Labels: $z_1$ prior earnings, $z_2$ motivation, $z_3$ household size, $z_4$ distance to the training centre; the other 96 are noise |
| Lottery offer | $W\mid Z \sim \text{Bernoulli}(\pi(Z))$, $\;\pi(z)=\Lambda(-0.2-0.8z_1+1.2z_3)$ |
| Resistance to training | $V\sim U[0,1]$, independent of $(Z,W)$ |
| Take-up | $D(w)=\mathbf 1\{V<p_w(Z)\}$ with $p_0(z)=0.10+0.15\,\mathbf 1\{z_1>1\}$ and $p_1(z)=p_0(z)+c(z)$, $\;c(z)=0.65\,\mathbf 1\{z_4<0\}+0.30\,\mathbf 1\{z_4\ge 0\}$ |
| Untreated earnings | $Y(0)=g_0(Z)-1.5\,(V-\tfrac12)+\varepsilon$, $\;g_0(z)=10+1.5z_1-0.5z_3+\mathbf 1\{z_6>0\}+0.5z_7z_8$, $\;\varepsilon\sim N(0,1.5^2)$ |
| Treated earnings | $Y(1)=Y(0)+\tau(Z)-2\,(V-\tfrac12)$, $\;\tau(z)=2+z_2$ |
| Observed | $D=WD(1)+(1-W)D(0)$, $\;Y=DY(1)+(1-D)Y(0)$ |

Default sample size: $n=4{,}000$.

### DGP-B — County adoption of a local policy, two periods plus a pre-period (Q3)

Counties choose whether to adopt a policy that takes effect at $t=1$. Outcomes are observed at $t\in\{-1,0,1\}$; the extra pre-period $t=-1$ is there only to run placebo tests.

| Component | Specification |
|---|---|
| County characteristics | $p=100$ covariates |
| Adoption | $D\mid Z\sim\text{Bernoulli}(\pi(Z))$, $\;\pi(z)=\Lambda\!\big(-0.5+0.8z_1+1.2\,\mathbf 1\{z_2>0\}\big)$ |
| Untreated outcomes | $Y_t(0)=a(Z)+\eta+\lambda_t+t\,m(Z)+\varepsilon_t$, with $a(z)=5+2z_1+z_5$, $\eta\sim N(0,1)$, $(\lambda_{-1},\lambda_0,\lambda_1)=(-0.3,0,0.3)$, $\varepsilon_t\overset{iid}{\sim}N(0,1)$ |
| County-specific trend | $m(z)=0.5+0.6z_1+1.2\,\mathbf 1\{z_1>0\}\mathbf 1\{z_2>0\}+0.5z_3z_4+0.8\max(z_1-1,0)$ |
| Treatment effect | $Y_1(1)=Y_1(0)+\tau$, $\;\tau=1+z_1+\nu$, $\nu\sim N(0,0.5^2)$; no effect before $t=1$ |
| Observed | $Y_{-1}=Y_{-1}(0)$, $Y_0=Y_0(0)$, $Y_1=DY_1(1)+(1-D)Y_1(0)$ |

Default sample size: $n=3{,}000$ counties. **DGP-B′ (limited overlap)** is identical except that every county with $z_1>1$ adopts: $\pi(z)=1$ if $z_1>1$.

### DGP-C — Staggered adoption (Q4)

A balanced panel of $N=2{,}000$ units over $T=8$ periods. Units adopt in period $G\in\{3,5,7\}$ or never ($G=\infty$), and stay treated.

| Component | Specification |
|---|---|
| Covariates | $p=50$ |
| Adoption cohort | $u=0.5z_1+0.8\,\mathbf 1\{z_2>0\}+\xi$, $\xi\sim\text{Logistic}(0,1)$; $\;G=3$ if $u>1.52$; $G=5$ if $0.63<u\le1.52$; $G=7$ if $-0.72<u\le0.63$; $G=\infty$ otherwise |
| Outcome | $Y_{it}=\alpha_i+0.2t+t\,m(Z_i)+\theta_{it}+e_{it}$, $\;\alpha_i=z_{1i}+N(0,1)$, $\;e_{it}\overset{iid}{\sim}N(0,1)$ |
| Treatment effect | $\theta_{it}=\delta_{G_i}\,(t-G_i+1)\,(1+0.2\zeta_{it})$ for $t\ge G_i$ and $0$ otherwise; $(\delta_3,\delta_5,\delta_7)=(0.5,0.3,0.1)$, $\zeta_{it}\overset{iid}{\sim}N(0,1)$ |
| Versions | **DGP-C0:** $m\equiv0$. **DGP-C1:** $m(z)=0.3z_1+0.4\,\mathbf 1\{z_2>0\}$. **DGP-C0-ant:** DGP-C0 plus an anticipation effect of $+0.3$ at $t=G_i-1$ for every eventually treated unit |

The true group-time ATT is $\theta_0(g,t)=\delta_g(t-g+1)$ for $t\ge g$.

## Setup

*Imports, seeds, and your implementation of DGP-A, DGP-B and DGP-C go here.*

---
## Question 1 — LATE with covariates: identification (slides 13–28)

Notation for Q1–Q2: $C\equiv\{D(1)=1,D(0)=0\}$ is the complier event, $\pi(z)\equiv P(W=1\mid Z=z)$, $\mu(w,z)\equiv E[Y\mid W=w,Z=z]$, $m(w,z)\equiv E[D\mid W=w,Z=z]$, and $\theta_{LATE}(z)\equiv E[Y(1)-Y(0)\mid C,Z=z]$.

### Q1(a) Identification, by hand

- Under the assumptions of slide 28 (conditional independence $\{Y(d),D(w)\}\perp W\mid Z$, overlap $0<\pi(Z)<1$, no defiers, and exclusion built into the notation $Y(d)$), show that for every $z$
$$\mu(1,z)-\mu(0,z)=\theta_{LATE}(z)\,P(C\mid Z=z)\qquad\text{and}\qquad m(1,z)-m(0,z)=P(C\mid Z=z).$$
Use these two identities to prove the slide-28 formula for $\theta_{LATE}$.
- Show that $\theta_{LATE}=E[\omega(Z)\,\theta_{LATE}(Z)]$ with $\omega(z)=P(C\mid Z=z)/P(C)$, and that $\omega$ is the density of $Z$ among compliers relative to the population: $E[f(Z)\mid C]=E[f(Z)\,\omega(Z)]$ for any function $f$.
- *Selection bias in the reduced form.* Show that when $W$ is random only given $Z$,
$$E[Y\mid W=1]-E[Y\mid W=0]=E\big[\mu(1,Z)-\mu(0,Z)\mid W=1\big]+\Big\{E[\mu(0,Z)\mid W=1]-E[\mu(0,Z)\mid W=0]\Big\},$$
and write the analogous decomposition of the first stage. Compare with the decomposition on slide 8.
- *DGP-A as a latent-index model.* Show that in DGP-A an applicant is an always-taker if and only if $V<p_0(Z)$, a complier if and only if $p_0(Z)\le V<p_1(Z)$, and a never-taker if and only if $V\ge p_1(Z)$, and that there are no defiers. Derive closed forms for $m(w,z)$, $\mu(w,z)$, $\theta_{LATE}(z)$, $\theta_{ATE}(z)\equiv E[Y(1)-Y(0)\mid Z=z]$, and the average effect for always-takers with $Z=z$. You will use $\mu$ and $m$ as **oracle** nuisances later.
- **Interpret:** The slide-28 estimand is a *ratio of averages*. A natural alternative is the *average of ratios*, $E[\theta_{LATE}(Z)]$. Which population does each describe, and when do they coincide? Using your closed forms, predict the ordering of $\theta_{ATT}$, $\theta_{LATE}$ and $\theta_{ATE}$ in DGP-A *before* computing them, and explain the role of the term $-2(V-\tfrac12)$. Why does the latent-index structure guarantee the uniformity assumption (Vytlacil, 2002)?

**Your answer — by hand (derivations):**

*Type your derivation here.*

**Your interpretation:**

*Write your interpretation here.*

### Q1(b) What each estimand recovers

- Simulate a population of 400,000 applicants from DGP-A. Using the simulated potential outcomes and types, report the four type shares, $\theta_{ATE}$, $\theta_{ATT}$, $\theta_{LATE}$, and $E[\theta_{LATE}(Z)]$ (from your closed form). Also report the population reduced form, first stage, and their ratio (the naive Wald estimand that ignores $Z$), and the population **2SLS** coefficient on $D$ from a regression of $Y$ on $D$ with $W$ as the instrument and all 100 covariates entered as additive linear controls (use the first 200,000 draws for 2SLS).
- Draw $R=200$ samples with $n=4{,}000$. In each, compute (i) the naive Wald estimator, (ii) 2SLS with additive linear controls, and (iii) the infeasible **oracle** slide-28 estimator that plugs your closed-form $\mu$ and $m$ into the sample averages. Report the Monte Carlo mean and standard deviation of each.
- **Interpret:**
  1. Explain each gap between $\theta_{ATE}$, $\theta_{ATT}$, $\theta_{LATE}$ and $E[\theta_{LATE}(Z)]$ in terms of *who is being averaged*.
  2. The population reduced form is close to zero although there are no defiers and $\theta_{LATE}\approx2$. Slide 18 produces a zero ITT for a different reason. Which reason, and which term of your Q1(a) decomposition is responsible here? Which covariates drive it, and in which direction?
  3. Does 2SLS with linear controls recover $\theta_{LATE}$? Why should you not expect it to in general (Blandhol et al., 2022)?
  4. The oracle estimator has a tiny standard deviation. What does that tell you about where the sampling noise of any *feasible* estimator of the slide-28 formula comes from?

**Your interpretation:**

*Write your interpretation here.*

### Q1(c) Who are the compliers? (trees)

- *By hand:* For $\kappa=1-\dfrac{D(1-W)}{1-\pi(Z)}-\dfrac{(1-D)W}{\pi(Z)}$, show that $E[\kappa\mid Z,\text{type}]=\mathbf 1\{\text{type}=C\}$, and conclude that $E[f(Z)\mid C]=E[\kappa f(Z)]/E[\kappa]$ (Abadie, 2003).
- On **one** sample ($n=4{,}000$), estimate $m(w,z)$ with a random-forest classifier fitted separately in each lottery arm. For applicants used to train a forest, use their **out-of-bag** prediction (the average over the trees whose bootstrap sample did not contain them); for the others, use the ordinary prediction. Form $\hat c(Z_i)=\hat m(1,Z_i)-\hat m(0,Z_i)$. Fit a regression tree of depth 2 to $\hat c(Z)$ on $Z$ and report its splits and leaf values. Fit the same depth-2 tree to the **true** $P(C\mid Z)$ and compare.
- Estimate complier means $E[z_j\mid C]$ for $j=1,\dots,8$ in two ways: with $\hat c$-weights (from Q1(a)) and with Abadie's $\kappa$ using $\pi$ estimated by $\ell_1$-penalized logistic regression. Compare with the population means $E[z_j]$, the true complier means (from simulated types), and the true always-taker means.
- **Interpret:** Who are the compliers? Does the tree built on $\hat c$ split on any covariate on which $P(C\mid Z)$ does not depend? If so, use the design of the lottery to explain where that split comes from. How can we describe compliers on average when no individual complier can ever be identified? Use your complier means and closed forms to explain the gap between $\theta_{LATE}$ and $\theta_{ATE}$ in Q1(b). Finally, a city plans to run the same lottery where *every* applicant lives far from the training centre ($z_4\ge0$). Using the population, compute $\theta_{LATE}$ among applicants with $z_4\ge0$ and with $z_4<0$. Explain the difference with the latent-index structure, and say what it implies for the external validity of a LATE.

**Your answer — by hand (derivations):**

*Type your derivation here.*

**Your interpretation:**

*Write your interpretation here.*

---
## Question 2 — LATE with high-dimensional nuisances: what ML gets right and wrong (slide 43)

Slide 43 says identification depends on nuisances that may be high dimensional, and that ML can estimate them flexibly. This question asks what happens to the **causal parameter** when you plug ML estimates of $\mu$, $m$ or $\pi$ into the identification formulas.

### Q2(a) Plug-in estimators and the source of their bias

- *By hand:* Let $\hat\mu_w$ be an estimate of $\mu(w,\cdot)$, treated as a fixed function, with error $e_w(z)=\hat\mu_w(z)-\mu(w,z)$. Show that the population plug-in numerator $E[\hat\mu_1(Z)-\hat\mu_0(Z)]$ has bias $E[e_1(Z)]-E[e_0(Z)]$. Then show that if each error averages to zero in its own arm, $E[e_w(Z)\mid W=w]=0$ (approximately true for an in-sample least-squares fit with an intercept), the bias equals
$$-\frac{\operatorname{Cov}\big(e_1(Z),\pi(Z)\big)}{E[\pi(Z)]}-\frac{\operatorname{Cov}\big(e_0(Z),\pi(Z)\big)}{E[1-\pi(Z)]}.$$
Which estimation errors are harmless for the causal parameter, and which are harmful?
- On one sample ($n=4{,}000$), estimate $\mu$ and $m$ separately in each lottery arm and form the slide-28 plug-in estimator with: (i) LASSO tuned by 5-fold cross-validation; (ii) LASSO with the BCCH (2012) plug-in penalty from Assignment 1 ($c=1.1$, $\gamma=0.1$, homoskedastic $\sigma$ updated twice), translated carefully into scikit-learn's normalization of the objective; (iii) random forests with $q\in\{\sqrt p,\,p/3,\,p\}$ candidate covariates per split (200 trees, at least 5 observations per leaf). Report numerator, denominator and LATE for each, next to the oracle on the same sample.
- For each learner, compute $e_w(Z_i)$ with your closed-form $\mu$. Report the within-arm mean error, the two covariance terms of your formula (using the true $\pi$), and the predicted versus actual numerator bias. For both LASSOs, report which covariates are selected and the coefficient on $z_3$ in each arm.
- **Interpret:** Rank the learners by bias. Explain, using your formula, why the "rigorous" BCCH penalty, designed to select the right covariates for *predicting* $Y$, gives the worst LASSO estimate of $\theta_{LATE}$. Why is $z_3$ a trap (Belloni, Chernozhukov and Hansen, 2014)? Why do random forests, which approximate nonlinear functions well, have such a large bias here, and why does it fall as $q$ rises (recall Assignment 2, Q3)? Is the damage mainly in the numerator or the denominator, and why?

**Your answer — by hand (derivations):**

*Type your derivation here.*

**Your interpretation:**

*Write your interpretation here.*

### Q2(b) Monte Carlo: is the bias large relative to the noise?

- *By hand:* If $\hat\theta\sim N(\theta+b,s^2)$, show that the interval $\hat\theta\pm1.96s$ covers $\theta$ with probability $\Phi(1.96-b/s)-\Phi(-1.96-b/s)$. Tabulate it for $b/s\in\{0,0.5,1,2,3\}$.
- Run $R=100$ replications with $n=4{,}000$ of: naive Wald, 2SLS with linear controls, oracle, LASSO-CV plug-in, LASSO-BCCH plug-in; and, in the first 30 replications only, the random-forest plug-in with $q=p/3$ (100 trees). For each estimator report the mean, bias, SD, RMSE, $|b|/s$, and the coverage of the *infeasible* interval $\hat\theta_r\pm1.96\,\text{SD}_{MC}$, both empirically and from your formula.
- Repeat the two LASSO estimators with $n=16{,}000$ ($R=50$).
- **Interpret:** Even with a perfect standard error, which estimators would deliver valid 95% intervals? Does quadrupling $n$ fix the LASSO problem? Compare how fast $b$ and $s$ shrink, and explain why a bias that vanishes as $n\to\infty$ can still invalidate inference at every sample size you will ever have. Why does this motivate estimators whose bias is only *second order* in the nuisance errors (Lecture 4)?

**Your answer — by hand (derivations):**

*Type your derivation here.*

**Your interpretation:**

*Write your interpretation here.*

### Q2(c) Weighting by the instrument propensity, and overfitting

- *By hand:* Under the slide-28 assumptions, show that
$$\theta_{LATE}=\frac{E\big[WY/\pi(Z)\big]-E\big[(1-W)Y/(1-\pi(Z))\big]}{E\big[WD/\pi(Z)\big]-E\big[(1-W)D/(1-\pi(Z))\big]}\qquad\text{(Frölich, 2007).}$$
In the sample version, normalize each arm's weights to sum to one (Hájek weights). Show that if the estimated propensities are $\hat\pi_i=1-\epsilon$ for every applicant with $W_i=1$ and $\hat\pi_i=\epsilon$ for every applicant with $W_i=0$, the normalized IPW estimator is exactly the naive Wald estimator, whatever the value of $\epsilon\in(0,1)$.
- On the same sample as Q2(a), compute the normalized IPW estimator with $\pi$ given by: (i) the truth; (ii) $\ell_1$-penalized logistic regression tuned by 5-fold CV, once with classification **accuracy** and once with **log-loss** as the CV criterion; (iii) a random-forest classifier's **in-sample** predicted probabilities; (iv) its **out-of-bag** probabilities; and (v) **5-fold out-of-fold** probabilities (fit on four folds, predict the fifth). Use 300 trees, at least 5 observations per leaf, and $q\in\{\sqrt p,\,p\}$ for (iii)–(iv), $q=\sqrt p$ for (v). Clip $\hat\pi$ to $[0.01,0.99]$ and report how many applicants are clipped. Plot $\hat\pi$ against the true $\pi$ for (iii) and (iv).
- Run a small Monte Carlo ($R=20$, $n=4{,}000$) for (i), (ii) with log-loss, and (iv) with $q=\sqrt p$.
- **Interpret:** Explain mechanically why the in-sample forest drives the IPW estimate toward the naive Wald ratio, linking your answer to the by-hand result. Out-of-bag and out-of-fold predictions remove that problem but not all of the bias. What is left, and is it the same phenomenon as in Q2(a)? Why does the CV criterion matter for a propensity model? Distinguish **overfitting bias** from **regularization bias**, using your results from Q2(a)–(c). Looking ahead: what two features should an estimator have to be robust to both?

**Your answer — by hand (derivations):**

*Type your derivation here.*

**Your interpretation:**

*Write your interpretation here.*

---
## Question 3 — Two-period DiD with high-dimensional covariates (slides 30–37, 41–43)

Write $\Delta Y\equiv Y_1-Y_0$, $\pi(z)\equiv P(D=1\mid Z=z)$ and $\mu_0(z)\equiv E[\Delta Y\mid D=0,Z=z]$.

### Q3(a) Identification, by hand

- Prove Theorem 3.1 (slide 35) from Assumptions 3.1 and 3.2.
- Suppose instead that parallel trends hold only conditionally, $E[Y_1(0)-Y_0(0)\mid D=1,Z]=E[Y_1(0)-Y_0(0)\mid D=0,Z]$, with no anticipation and $\pi(Z)<1$. Prove
$$\theta_{ATT}=E\big[\Delta Y-\mu_0(Z)\mid D=1\big]\qquad\text{(Heckman, Ichimura and Todd, 1997)},$$
the two-period case of slide 42. Then prove the weighting representation
$$\theta_{ATT}=\frac{1}{P(D=1)}\,E\!\left[\frac{D-\pi(Z)}{1-\pi(Z)}\,\Delta Y\right]\qquad\text{(Abadie, 2005)}.$$
Why does the ATT require $\pi(Z)<1$ but not $\pi(Z)>0$?
- Show that in a two-period panel, OLS of $\Delta Y$ on $(1,D,Z)$ gives the same coefficient on $D$ as a TWFE regression of $Y_{it}$ on unit effects, a period effect, $D_i\times\text{Post}_t$ and $Z_i\times\text{Post}_t$.
- In DGP-B, show that the unconditional DiD estimand equals $\theta_{ATT}+E[m(Z)\mid D=1]-E[m(Z)\mid D=0]$, and that the placebo DiD that replaces $(Y_0,Y_1)$ by $(Y_{-1},Y_0)$ estimates exactly the bias term. Show that, for any fixed function $\hat\mu_0$, the regression-adjusted (outcome-regression) DiD has bias $E[\mu_0(Z)-\hat\mu_0(Z)\mid D=1]$, and explain why in DGP-B its placebo version is centred on the same bias when the same learner is used in both periods.

**Your answer — by hand (derivations):**

*Type your derivation here.*

**Your interpretation:**

*Write your interpretation here.*

### Q3(b) Estimation and the placebo test

- Using a population of 300,000 counties, report $\theta_{ATT}$, $\theta_{ATE}$, the unconditional DiD estimand, the OLS coefficient on $D$ in the regression of $\Delta Y$ on $(1,D,Z)$ (use the first 150,000 counties), and the OLS coefficient when the 100 covariates are replaced by the single oracle control $m(Z)$.
- On one sample ($n=3{,}000$), compute each estimator below twice: on the **post** change $Y_1-Y_0$ and on the **placebo** change $Y_0-Y_{-1}$: (i) unconditional DiD; (ii) OLS of the change on $(1,D,Z)$; (iii) outcome-regression DiD with $\hat\mu_0$ from a CV-tuned LASSO fitted on control counties; (iv) the same with a random forest ($q=p$, 200 trees, at least 5 observations per leaf); (v) normalized IPW-DiD (Abadie, 2005) with $\hat\pi$ from log-loss-tuned $\ell_1$-logistic regression and from the out-of-bag probabilities of a random-forest classifier; (vi) the oracle outcome-regression and IPW estimators using the true $\mu_0$ and $\pi$.
- **Interpret:** Which estimators pass the placebo test? Does each estimator's placebo estimate predict its own bias? OLS passes the placebo yet misses $\theta_{ATT}$: what does it estimate instead, and why (Angrist, 1998; Słoczyński, 2022)? Explain why the forest beats LASSO for $\mu_0$, using the shape of $m$ in DGP-B. Neither propensity model is correctly specified: which feature of $\pi$ does each one get wrong, and does it matter much here? Give one situation in which passing a placebo test should *not* reassure you (Roth, 2022).

**Your interpretation:**

*Write your interpretation here.*

### Q3(c) Overlap and extrapolation: trees cannot extrapolate

- Using a population of 300,000 counties from DGP-B′, report $\theta_{ATT}$, the ATT of treated counties with $z_1\le1$ (the *trimmed* ATT), and the share of treated counties with $z_1>1$.
- On one sample ($n=3{,}000$) from DGP-B′, estimate $\pi$ with log-loss-tuned $\ell_1$-logistic regression and with out-of-bag random-forest probabilities; plot the estimated propensities by treatment status. Then compute the outcome-regression DiD with LASSO and with a forest (as in Q3(b)) and the IPW-DiD, both for all treated counties and for treated counties with $z_1\le1$. Compare each with the matching true ATT.
- For treated counties, plot the binned mean of the error $\hat\mu_0(Z_i)-\mu_0(Z_i)$ against $z_{1i}$ for both learners, marking $z_1=1$.
- **Interpret:** Why can no estimator recover $\theta_{ATT}$ in DGP-B′, and which assumption of slide 41 fails? Describe the shape of each learner's error beyond $z_1=1$ and explain it from how a forest and a LASSO form predictions. Is the LASSO's linear extrapolation "better"? Is trimming a fix, or a change of question (Crump et al., 2009)? Do your estimated propensity scores reveal that $\pi(z)=1$ for some counties? Why or why not?

**Your interpretation:**

*Write your interpretation here.*

---
## Question 4 — Staggered adoption: GT-ATTs, event studies, and covariates (slides 38–43)

Write $D_{it}=\mathbf 1\{t\ge G_i\}$ and use the not-yet-treated comparison set $C_{gt}=\{s\in\mathcal G:s>g\vee t\}$ of slide 40, which includes $G=\infty$.

### Q4(a) What does TWFE estimate under staggered adoption?

- *By hand:* Take three periods, an early cohort ($g=2$) and a late cohort ($g=3$), no never-treated units, no anticipation and parallel trends. Show that the 2×2 DiD that compares the late cohort with the early cohort between $t=2$ and $t=3$ identifies $\theta_0(3,3)-\{\theta_0(2,3)-\theta_0(2,2)\}$ (Goodman-Bacon, 2021). Give a numerical example in which every $\theta_0(g,t)>0$ but this comparison is negative.
- *By hand:* Using the Frisch–Waugh–Lovell theorem, show that the static TWFE coefficient is $\hat\beta=\sum_{i,t}\tilde D_{it}Y_{it}/\sum_{i,t}\tilde D_{it}D_{it}$, where $\tilde D$ is the residual from regressing $D_{it}$ on unit and time dummies, and that in a balanced panel $\tilde D_{it}=D_{it}-\bar D_{i\cdot}-\bar D_{\cdot t}+\bar D$. Conclude that under parallel trends and no anticipation, $E[\hat\beta\mid G]=\sum_{i,t}w_{it}\theta_{it}$ with $w_{it}=\tilde D_{it}D_{it}/\sum_{j,s}\tilde D_{js}D_{js}$ (de Chaisemartin and D'Haultfœuille, 2020), and that the weights sum to one.
- On one sample of DGP-C0, estimate the static TWFE coefficient. Compute the weights $w_{it}$ and report: their sums by cohort and period, the share of treated unit-periods with negative weight, and the check $\sum w_{it}\theta_{it}$ against $\hat\beta$ (using the true $\theta_{it}$). Compare $\hat\beta$ with the average effect over all treated unit-periods. Then recompute the cohort weight sums for a hypothetical design in which each cohort's effect is constant over time.
- **Interpret:** Which unit-periods receive negative weight, and why does this connect to your first derivation? Why does TWFE understate the average effect in DGP-C0? Would TWFE be a positively weighted average of cohort effects if effects did not change with time since adoption? Would those weights equal the cohort shares?

**Your answer — by hand (derivations):**

*Type your derivation here.*

**Your interpretation:**

*Write your interpretation here.*

### Q4(b) GT-ATTs and the event study (Callaway and Sant'Anna, 2021)

- On the same DGP-C0 sample, implement
$$\hat\theta(g,t)=\frac{1}{N_g}\sum_{i:G_i=g}(Y_{it}-Y_{i,g-1})-\frac{1}{N_{C_{gt}}}\sum_{i:G_i\in C_{gt}}(Y_{it}-Y_{i,g-1})$$
for $g\in\{3,5,7\}$ and every $t\in\{1,\dots,8\}$ except the base period $t=g-1$. Tabulate $\hat\theta(g,t)$ against $\theta_0(g,t)$.
- Aggregate to the event-study parameter $\hat\theta_{es}(e)$ of slide 39 for $e\in\{-6,\dots,5\}$, and to the average effect over all treated unit-periods. Compute pointwise 95% confidence intervals with a unit-level (cluster) bootstrap, $B=199$.
- Estimate the dynamic TWFE event study: regress $Y_{it}$ on unit and time effects and relative-time dummies $\mathbf 1\{t-G_i=e\}$ for $e\in\{-6,\dots,5\}$, $e\ne-1$. Plot both event studies with the truth.
- **Interpret:** Which $(g,t)$ cells enter $\hat\theta_{es}(5)$, $\hat\theta_{es}(0)$ and $\hat\theta_{es}(-6)$? Why does the cohort composition change with $e$, and what does that imply for reading an event-study plot as a "dynamic path"? The dynamic TWFE regression shows non-zero pre-period coefficients although parallel trends hold by construction. Explain why (Sun and Abraham, 2021). What would a researcher who "tests for pre-trends" with that regression conclude?

**Your interpretation:**

*Write your interpretation here.*

### Q4(c) Covariates and anticipation

- On one sample of DGP-C1, compute the unconditional event study from Q4(b). Then compute the covariate-adjusted version from slide 42: for each $(g,t)$ cell, fit $\hat\mu_{gt}(z)\approx E[Y_t-Y_{g-1}\mid G\in C_{gt},Z=z]$ on the comparison units with (i) a CV-tuned LASSO and (ii) a random forest ($q=p$, 200 trees, at least 5 observations per leaf), and set $\hat\theta(g,t)=N_g^{-1}\sum_{i:G_i=g}\{Y_{it}-Y_{i,g-1}-\hat\mu_{gt}(Z_i)\}$. Aggregate and plot the three event studies with the truth.
- *By hand:* Why must $\hat\mu_{gt}$ be re-estimated for every $(g,t)$ rather than once? State the overlap condition the $(g,t)$ comparison actually needs, in terms of $P(G=g\mid Z)$ and $P(G\in C_{gt}\mid Z)$.
- On one sample of DGP-C0-ant, compute the unconditional event study twice: with base period $g-1$ and comparison set $C_{gt}$ (as in Q4(b)), and with base period $g-2$ and comparison set $\{s\in\mathcal G:s>(g\vee t)+1\}$.
- **Interpret:** Which learner gives flat pre-period estimates and the most accurate post-period estimates, and why, given the shape of $m$? Why do the errors of the unconditional event study grow with $e$? With anticipation, describe the pattern produced by base period $g-1$, explain how moving the base period and shrinking the comparison set fix it, and state the cost. Which assumption on slide 41 is at stake?

**Your answer — by hand (derivations):**

*Type your derivation here.*

**Your interpretation:**

*Write your interpretation here.*

---
## References

- Abadie, A. (2003). Semiparametric instrumental variable estimation of treatment response models. *Journal of Econometrics*, 113(2), 231–263.
- Abadie, A. (2005). Semiparametric difference-in-differences estimators. *Review of Economic Studies*, 72(1), 1–19.
- Angrist, J. D. (1998). Estimating the labor market impact of voluntary military service using Social Security data on military applicants. *Econometrica*, 66(2), 249–288.
- Baker, A., Callaway, B., Cunningham, S., Goodman-Bacon, A., and Sant'Anna, P. H. C. (2026). Difference-in-differences designs: A practitioner's guide. *Journal of Economic Literature*, 64, 498–557.
- Belloni, A., Chen, D., Chernozhukov, V., and Hansen, C. (2012). Sparse models and methods for optimal instruments with an application to eminent domain. *Econometrica*, 80(6), 2369–2429.
- Belloni, A., Chernozhukov, V., and Hansen, C. (2014). Inference on treatment effects after selection among high-dimensional controls. *Review of Economic Studies*, 81(2), 608–650.
- Blandhol, C., Bonney, J., Mogstad, M., and Torgovitsky, A. (2022). When is TSLS actually LATE? NBER Working Paper 29709.
- Breiman, L. (2001). Random forests. *Machine Learning*, 45(1), 5–32.
- Callaway, B., and Sant'Anna, P. H. C. (2021). Difference-in-differences with multiple time periods. *Journal of Econometrics*, 225(2), 200–230.
- Chernozhukov, V., Chetverikov, D., Demirer, M., Duflo, E., Hansen, C., Newey, W., and Robins, J. (2018). Double/debiased machine learning for treatment and structural parameters. *Econometrics Journal*, 21(1), C1–C68.
- Chernozhukov, V., Hansen, C., Kallus, N., Spindler, M., and Syrgkanis, V. (2024). *Applied Causal Inference Powered by ML and AI*. Online manuscript.
- Crump, R. K., Hotz, V. J., Imbens, G. W., and Mitnik, O. A. (2009). Dealing with limited overlap in estimation of average treatment effects. *Biometrika*, 96(1), 187–199.
- de Chaisemartin, C., and D'Haultfœuille, X. (2020). Two-way fixed effects estimators with heterogeneous treatment effects. *American Economic Review*, 110(9), 2964–2996.
- Finkelstein, A., Taubman, S., Wright, B., Bernstein, M., Gruber, J., Newhouse, J. P., Allen, H., Baicker, K., and the Oregon Health Study Group (2012). The Oregon Health Insurance Experiment: Evidence from the first year. *Quarterly Journal of Economics*, 127(3), 1057–1106.
- Frölich, M. (2007). Nonparametric IV estimation of local average treatment effects with covariates. *Journal of Econometrics*, 139(1), 35–75.
- Goodman-Bacon, A. (2021). Difference-in-differences with variation in treatment timing. *Journal of Econometrics*, 225(2), 254–277.
- Heckman, J. J., Ichimura, H., and Todd, P. E. (1997). Matching as an econometric evaluation estimator: Evidence from evaluating a job training programme. *Review of Economic Studies*, 64(4), 605–654.
- Heckman, J. J., and Vytlacil, E. (2005). Structural equations, treatment effects, and econometric policy evaluation. *Econometrica*, 73(3), 669–738.
- Hirano, K., Imbens, G. W., and Ridder, G. (2003). Efficient estimation of average treatment effects using the estimated propensity score. *Econometrica*, 71(4), 1161–1189.
- Imbens, G. W., and Angrist, J. D. (1994). Identification and estimation of local average treatment effects. *Econometrica*, 62(2), 467–475.
- Roth, J. (2022). Pretest with caution: Event-study estimates after testing for parallel trends. *American Economic Review: Insights*, 4(3), 305–322.
- Sant'Anna, P. H. C., and Zhao, J. (2020). Doubly robust difference-in-differences estimators. *Journal of Econometrics*, 219(1), 101–122.
- Słoczyński, T. (2022). Interpreting OLS estimands when treatment effects are heterogeneous: Smaller groups get larger weights. *Review of Economics and Statistics*, 104(3), 501–509.
- Sun, L., and Abraham, S. (2021). Estimating dynamic treatment effects in event studies with heterogeneous treatment effects. *Journal of Econometrics*, 225(2), 175–199.
- Vytlacil, E. (2002). Independence, monotonicity, and latent index models: An equivalence result. *Econometrica*, 70(1), 331–341.

## AI-use acknowledgment

*Insert the acknowledgment statement required by the course README here.*